# 🫀 3D Medical Image Segmentation with nnFormer (12 Swin-Transformer Encoder Blocks)
### End-to-End Pipeline for ACDC Cardiac MRI & Synapse Multi-Organ CT

This notebook provides a complete pipeline to prepare datasets, preprocess 3D volumes, train, and evaluate **nnFormer** on Kaggle GPUs (T4 / P100 / A100).

#### Datasets & Paper-Specified Splits Included:
1. **ACDC (Cardiac MRI)**: 100 Patients total $\rightarrow$ **70 Training**, **10 Validation**, **20 Testing** (3 structures: RV, Myocardium, LV).
2. **Synapse Multi-Organ (Abdominal CT)**: 30 Cases total $\rightarrow$ **18 Training**, **12 Testing** (TransUNet split: 8 organs - aorta, gallbladder, spleen, left kidney, right kidney, liver, pancreas, stomach).

> 💡 **Tip:** Enable GPU accelerator in Kaggle Notebook settings (**Settings -> Accelerator -> GPU T4 x2 or P100**).

In [ ]:
# 1. Environment Configuration
import os
import sys
import torch

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))

# Configure nnFormer Environment Paths & PYTHONPATH
BASE_DIR = '/kaggle/working'
PROJECT_DIR = os.path.join(BASE_DIR, 'nextgen-transformer')
DATASET_DIR = os.path.join(BASE_DIR, 'DATASET')

os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['PYTHONPATH'] = PROJECT_DIR + ':' + os.environ.get('PYTHONPATH', '')
os.environ['nnFormer_raw_data_base'] = os.path.join(DATASET_DIR, 'nnFormer_raw')
os.environ['nnFormer_preprocessed'] = os.path.join(DATASET_DIR, 'nnFormer_preprocessed')
os.environ['RESULTS_FOLDER'] = os.path.join(DATASET_DIR, 'nnFormer_trained_models')
os.environ['nnFormer_def_n_proc'] = '2'
os.environ['nnFormer_n_proc_DA'] = '1'
os.environ['MAX_NUM_EPOCHS'] = '250'
os.environ['NUM_BATCHES_PER_EPOCH'] = '250'
os.environ['NUM_VAL_BATCHES_PER_EPOCH'] = '50'

print('\n--- Configured Paths ---')
print('PYTHONPATH:', os.environ['PYTHONPATH'])
print('nnFormer_raw_data_base:', os.environ['nnFormer_raw_data_base'])
print('nnFormer_preprocessed:', os.environ['nnFormer_preprocessed'])
print('RESULTS_FOLDER:', os.environ['RESULTS_FOLDER'])

In [ ]:
# 2. Clone Fresh Repository & Install Dependencies
%cd /kaggle/working
import sys
import os
import numpy as np

if np.__version__.startswith('2.'):
    print('Updating NumPy to 1.x for C-ABI compatibility...')
    !pip install -q "numpy<2" batchgenerators==0.23 SimpleITK nibabel timm einops medpy reportlab
    print('Restarting Python kernel automatically...')
    os._exit(0)

!rm -rf /kaggle/working/nextgen-transformer
!git clone https://github.com/sharadpawarsaini/nextgen-transformer.git /kaggle/working/nextgen-transformer
%cd /kaggle/working/nextgen-transformer

!pip install -q batchgenerators==0.23 SimpleITK nibabel timm einops medpy reportlab
!pip install -e . --no-build-isolation

PROJECT_DIR = '/kaggle/working/nextgen-transformer'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('✅ Repository cloned and dependencies installed successfully!')

In [ ]:
# 3. Dataset Setup: ACDC (70/10/20 Split) & Synapse (18/12 TransUNet Split)
import subprocess

print('Setting up ACDC (100 patients) and Synapse Multi-Organ (30 cases) datasets...')
PROJECT_DIR = '/kaggle/working/nextgen-transformer'
subprocess.run([sys.executable, 'prepare_full_acdc_synapse.py'], cwd=PROJECT_DIR, check=True)
print('✅ Datasets generated and saved successfully!')

In [ ]:
# 4. Preprocessing: Task001_ACDC (Cardiac MRI) & Task002_Synapse (Abdominal CT)
import os
import sys
import subprocess

PROJECT_DIR = '/kaggle/working/nextgen-transformer'

print('1. Preprocessing Task001_ACDC (Cardiac MRI)...')
cmd_acdc = [sys.executable, '-m', 'nnformer.experiment_planning.nnFormer_plan_and_preprocess', '-t', '1', '-tl', '2', '-tf', '2', '--verify_dataset_integrity']
subprocess.run(cmd_acdc, env=os.environ, cwd=PROJECT_DIR, check=True)

print('2. Preprocessing Task002_Synapse (Abdominal CT)...')
cmd_synapse = [sys.executable, '-m', 'nnformer.experiment_planning.nnFormer_plan_and_preprocess', '-t', '2', '-tl', '2', '-tf', '2', '--verify_dataset_integrity']
subprocess.run(cmd_synapse, env=os.environ, cwd=PROJECT_DIR, check=True)

print('✅ Preprocessing completed for both datasets!')

In [ ]:
# 5. Train nnFormer Models (12 Swin-Transformer Encoder Blocks)
import os
import sys
import subprocess

PROJECT_DIR = '/kaggle/working/nextgen-transformer'

print('--- Training Task001_ACDC (Cardiac MRI - 12 Encoder Blocks) ---')
cmd_acdc_train = [sys.executable, '-m', 'nnformer.run.run_training', '3d_fullres', 'nnFormerTrainerV2_nnformer_acdc', '1', '0']
subprocess.run(cmd_acdc_train, env=os.environ, cwd=PROJECT_DIR, check=True)

print('--- Training Task002_Synapse (Abdominal CT Multi-Organ - 12 Encoder Blocks) ---')
cmd_synapse_train = [sys.executable, '-m', 'nnformer.run.run_training', '3d_fullres', 'nnFormerTrainerV2_nnformer_synapse', '2', '0']
subprocess.run(cmd_synapse_train, env=os.environ, cwd=PROJECT_DIR, check=True)

print('✅ Training completed for both ACDC and Synapse!')

In [ ]:
# 6. Perform Inference on Test Sets
import os
import sys
import subprocess

PROJECT_DIR = '/kaggle/working/nextgen-transformer'
RAW_DATA = os.path.join(os.environ['nnFormer_raw_data_base'], 'nnFormer_raw_data')

# 1. ACDC Test Inference
acdc_task_dir = os.path.join(RAW_DATA, 'Task001_ACDC')
acdc_out = os.path.join(acdc_task_dir, 'inferTs', 'nnformer_acdc')
os.makedirs(acdc_out, exist_ok=True)
cmd_acdc_infer = [sys.executable, '-m', 'nnformer.inference.predict_simple', '-i', os.path.join(acdc_task_dir, 'imagesTs'), '-o', acdc_out, '-m', '3d_fullres', '-t', '1', '-tr', 'nnFormerTrainerV2_nnformer_acdc', '-chk', 'model_final_checkpoint']
subprocess.run(cmd_acdc_infer, env=os.environ, cwd=PROJECT_DIR, check=True)

# 2. Synapse Test Inference
synapse_task_dir = os.path.join(RAW_DATA, 'Task002_Synapse')
synapse_out = os.path.join(synapse_task_dir, 'inferTs', 'nnformer_synapse')
os.makedirs(synapse_out, exist_ok=True)
cmd_synapse_infer = [sys.executable, '-m', 'nnformer.inference.predict_simple', '-i', os.path.join(synapse_task_dir, 'imagesTs'), '-o', synapse_out, '-m', '3d_fullres', '-t', '2', '-tr', 'nnFormerTrainerV2_nnformer_synapse', '-chk', 'model_final_checkpoint']
subprocess.run(cmd_synapse_infer, env=os.environ, cwd=PROJECT_DIR, check=True)

print('✅ Test inference complete for both ACDC and Synapse!')

In [ ]:
# 7. Calculate Dice Similarity Coefficient (DSC) Metrics
import os
import glob
import numpy as np
import nibabel as nib

def calculate_dice(pred, gt, class_id):
    pred_c = (pred == class_id)
    gt_c = (gt == class_id)
    intersection = np.logical_and(pred_c, gt_c).sum()
    denominator = pred_c.sum() + gt_c.sum()
    if denominator == 0:
        return 1.0
    return (2.0 * intersection) / denominator

RAW_DATA = os.path.join(os.environ['nnFormer_raw_data_base'], 'nnFormer_raw_data')

# ========================================================
# 🫀 1. ACDC CARDIAC MRI EVALUATION (20 TEST PATIENTS)
# ========================================================
acdc_infer_dir = os.path.join(RAW_DATA, 'Task001_ACDC', 'inferTs', 'nnformer_acdc')
acdc_gt_dir = os.path.join(RAW_DATA, 'Task001_ACDC', 'labelsTs')
acdc_test_files = sorted(glob.glob(os.path.join(acdc_gt_dir, '*.nii.gz')))

if acdc_test_files:
    rv_scores, myo_scores, lv_scores = [], [], []
    for gt_path in acdc_test_files:
        fname = os.path.basename(gt_path)
        pred_path = os.path.join(acdc_infer_dir, fname)
        if os.path.exists(pred_path):
            p = nib.load(pred_path).get_fdata()
            g = nib.load(gt_path).get_fdata()
            rv_scores.append(calculate_dice(p, g, 1) * 100)
            myo_scores.append(calculate_dice(p, g, 2) * 100)
            lv_scores.append(calculate_dice(p, g, 3) * 100)
    
    if rv_scores:
        mean_rv = np.mean(rv_scores)
        mean_myo = np.mean(myo_scores)
        mean_lv = np.mean(lv_scores)
        acdc_overall_mean = (mean_rv + mean_myo + mean_lv) / 3.0
        print('=' * 65)
        print('       🫀 ACDC CARDIAC MRI EVALUATION RESULTS (12 BLOCKS)')
        print('=' * 65)
        print(f'  Right Ventricle (RV) Mean DSC : {mean_rv:.2f}%')
        print(f'  Myocardium (Myo) Mean DSC     : {mean_myo:.2f}%')
        print(f'  Left Ventricle (LV) Mean DSC  : {mean_lv:.2f}%')
        print('-' * 65)
        print(f'  🏆 ACDC OVERALL MEAN DICE ACCURACY : {acdc_overall_mean:.2f}%')
        print('=' * 65 + '\n\n')

# ========================================================
# 🩺 2. SYNAPSE MULTI-ORGAN CT EVALUATION (12 TEST CASES)
# ========================================================
syn_infer_dir = os.path.join(RAW_DATA, 'Task002_Synapse', 'inferTs', 'nnformer_synapse')
syn_gt_dir = os.path.join(RAW_DATA, 'Task002_Synapse', 'labelsTs')
syn_test_files = sorted(glob.glob(os.path.join(syn_gt_dir, '*.nii.gz')))

if syn_test_files:
    organs = ['Aorta', 'Gallbladder', 'Spleen', 'Left Kidney', 'Right Kidney', 'Liver', 'Pancreas', 'Stomach']
    organ_scores = {o: [] for o in organs}
    for gt_path in syn_test_files:
        fname = os.path.basename(gt_path)
        pred_path = os.path.join(syn_infer_dir, fname)
        if os.path.exists(pred_path):
            p = nib.load(pred_path).get_fdata()
            g = nib.load(gt_path).get_fdata()
            for idx, organ in enumerate(organs, start=1):
                organ_scores[organ].append(calculate_dice(p, g, idx) * 100)
    
    if organ_scores['Aorta']:
        print('=' * 65)
        print('   🩺 SYNAPSE MULTI-ORGAN CT EVALUATION RESULTS (12 BLOCKS)')
        print('=' * 65)
        organ_means = []
        for organ in organs:
            m_score = np.mean(organ_scores[organ])
            organ_means.append(m_score)
            print(f'  {organ:<22} Mean DSC : {m_score:.2f}%')
        synapse_overall_mean = np.mean(organ_means)
        print('-' * 65)
        print(f'  🏆 SYNAPSE 8-ORGAN OVERALL MEAN DICE ACCURACY : {synapse_overall_mean:.2f}%')
        print('=' * 65)

In [ ]:
# 8. Visualizing Results (ACDC Cardiac MRI & Synapse Multi-Organ CT)
import matplotlib.pyplot as plt
import nibabel as nib

RAW_DATA = os.path.join(os.environ['nnFormer_raw_data_base'], 'nnFormer_raw_data')

# Plot ACDC Slice
acdc_img = nib.load(os.path.join(RAW_DATA, 'Task001_ACDC', 'imagesTs', 'patient081_0000.nii.gz')).get_fdata()
acdc_gt = nib.load(os.path.join(RAW_DATA, 'Task001_ACDC', 'labelsTs', 'patient081.nii.gz')).get_fdata()
acdc_pred = nib.load(os.path.join(RAW_DATA, 'Task001_ACDC', 'inferTs', 'nnformer_acdc', 'patient081.nii.gz')).get_fdata()
s_idx = acdc_img.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(acdc_img[s_idx], cmap='gray'); axes[0].set_title('ACDC Input MRI Slice'); axes[0].axis('off')
axes[1].imshow(acdc_gt[s_idx], cmap='jet'); axes[1].set_title('ACDC Ground Truth'); axes[1].axis('off')
axes[2].imshow(acdc_pred[s_idx], cmap='jet'); axes[2].set_title('nnFormer ACDC Prediction'); axes[2].axis('off')
plt.tight_layout(); plt.show()

# Plot Synapse Slice
syn_img = nib.load(os.path.join(RAW_DATA, 'Task002_Synapse', 'imagesTs', 'case0019_0000.nii.gz')).get_fdata()
syn_gt = nib.load(os.path.join(RAW_DATA, 'Task002_Synapse', 'labelsTs', 'case0019.nii.gz')).get_fdata()
syn_pred = nib.load(os.path.join(RAW_DATA, 'Task002_Synapse', 'inferTs', 'nnformer_synapse', 'case0019.nii.gz')).get_fdata()
s_idx_syn = syn_img.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(syn_img[s_idx_syn], cmap='gray'); axes[0].set_title('Synapse Input CT Slice'); axes[0].axis('off')
axes[1].imshow(syn_gt[s_idx_syn], cmap='jet'); axes[1].set_title('Synapse Ground Truth'); axes[1].axis('off')
axes[2].imshow(syn_pred[s_idx_syn], cmap='jet'); axes[2].set_title('nnFormer Synapse Prediction'); axes[2].axis('off')
plt.tight_layout(); plt.show()